# 01: Pull and Clean the DOHMH Restaurant Inspection Data

General-purpose cleaning step: reads the raw NYC DOHMH Restaurant Inspection Results CSV, drops exact duplicate rows, standardizes column names, parses inspection dates, and cleans zip codes -- then writes the result so the rest of the pipeline (`02_analysis.ipynb`, `04_grade_cutoff_bunching.ipynb`) can read a single clean file instead of repeating this cleaning themselves.

**Takes in:** `data/DOHMH_New_York_City_Restaurant_Inspection_Results.csv` ( https://www.kaggle.com/datasets/new-york-city/nyc-inspections?resource=download ))

**Outputs:** `data/cleaned_inspections.csv`

In [1]:
import pandas as pd

import utils

USECOLS = [
    "CAMIS", "DBA", "BORO", "ZIPCODE", "CUISINE DESCRIPTION",
    "INSPECTION DATE", "SCORE", "GRADE",
]

## Load raw data and drop exact duplicate rows

In [2]:
df = pd.read_csv(utils.DATA_PATH, usecols=USECOLS)
print(f"Rows loaded: {len(df):,}")

df = df.drop_duplicates()
print(f"Rows after dropping exact duplicate rows: {len(df):,}")

Rows loaded: 399,918
Rows after dropping exact duplicate rows: 154,587


## Standardize column names

In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.head()

,camis,dba,boro,zipcode,cuisine_description,inspection_date,score,grade
0,40511702,NOTARO RESTAURANT,MANHATTAN,10016.0,Italian,06/15/2015,30.0,NaN
1,40511702,NOTARO RESTAURANT,MANHATTAN,10016.0,Italian,11/25/2014,NaN,NaN
2,50046354,VITE BAR,QUEENS,11106.0,Italian,10/03/2016,2.0,NaN
3,50061389,TACK'S CHINESE TAKE OUT,STATEN ISLAND,10314.0,Chinese,05/17/2017,46.0,NaN
4,41516263,NO QUARTER,BROOKLYN,11209.0,American,03/30/2017,18.0,NaN


## Parse inspection dates and drop placeholder rows

DOHMH uses a placeholder date (year 1900) for records that haven't actually been inspected yet; those aren't real observations, so they're dropped here.

In [4]:
df["inspection_date"] = pd.to_datetime(df["inspection_date"], errors="coerce")
n_before = len(df)
df = df[df["inspection_date"].dt.year > 1900]
print(f"Rows before dropping placeholder inspection dates: {n_before:,}")
print(f"Rows after dropping placeholder inspection dates: {len(df):,}")

Rows before dropping placeholder inspection dates: 154,587
Rows after dropping placeholder inspection dates: 153,452


## Clean zip codes and scores

In [5]:
n_before = len(df)
df = utils.clean_zipcode(df, zip_col="zipcode")
print(f"Rows before zip code cleaning: {n_before:,}")
print(f"Rows after zip code cleaning (unparseable zip codes dropped): {len(df):,}")

df["score"] = pd.to_numeric(df["score"], errors="coerce")

Rows before zip code cleaning: 153,452
Rows after zip code cleaning (unparseable zip codes dropped): 153,452


## Save cleaned data

In [6]:
utils.CLEANED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(utils.CLEANED_DATA_PATH, index=False)
print(f"Saved {len(df):,} rows to {utils.CLEANED_DATA_PATH}")
df.head()

Saved 153,452 rows to /Users/ethan/qss20-nyc-restaurant-inspections/data/cleaned_inspections.csv


,camis,dba,boro,zipcode,cuisine_description,inspection_date,score,grade
0,40511702,NOTARO RESTAURANT,MANHATTAN,10016,Italian,2015-06-15,30.0,NaN
1,40511702,NOTARO RESTAURANT,MANHATTAN,10016,Italian,2014-11-25,NaN,NaN
2,50046354,VITE BAR,QUEENS,11106,Italian,2016-10-03,2.0,NaN
3,50061389,TACK'S CHINESE TAKE OUT,STATEN ISLAND,10314,Chinese,2017-05-17,46.0,NaN
4,41516263,NO QUARTER,BROOKLYN,11209,American,2017-03-30,18.0,NaN
